In [ ]:
import os
import sys
import yaml
import pandas as pd
import numpy as np
import json
import re

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']
DATA_PATH = local_config['DATA_PATH']

sys.path.append(os.path.join(LOCAL_PATH, "src/python"))

rng = np.random.default_rng(seed=42)

In [7]:
# helper function for sorting agenda items
def sort_key(d):
    m = re.match(r'(\d+)(\D*)', d['item_no'])
    return (int(m.group(1)), m.group(2))

In [8]:
meetings_df = pd.read_csv(os.path.join(DATA_PATH, "intermediate_data/cpc/meetings-manifest.csv"))
DATES = sorted(list(meetings_df['date']))

In [9]:
MIN_IDX = 0
MAX_IDX = 250

In [10]:
for idx, row in meetings_df.iterrows():
    if idx < MIN_IDX or idx > MAX_IDX:
        continue

    date = row['date']
    year = date[0:4]

    print(f"{idx} {row['date']} ...")

    # Input data
    input_filepath = os.path.join(DATA_PATH, "intermediate_data/cpc", year, date, "agenda.txt")
    override_filepath = os.path.join(LOCAL_PATH, "intermediate_data/cpc", year, date, "agenda-manual-override.txt")
    if (not os.path.exists(input_filepath)) and (not os.path.exists(override_filepath)):
        print("No agenda text file found, skipping.")
        continue
    if os.path.exists(override_filepath):
        with open(override_filepath, 'r', encoding='utf-8') as f:
            text = f.read()
    else:
        with open(input_filepath, 'r', encoding='utf-8') as f:
            text = f.read()

    # Output paths
    output_dir = os.path.join(DATA_PATH, "intermediate_data/cpc", year, date)
    os.makedirs(output_dir, exist_ok=True)
    output_filepath = os.path.join(output_dir, "agenda-cleaned.json")

    # Clean
    lines = text.replace("\r\n", "\n").split("\n")

    # Find where agenda items start
    first_item_idx = None
    for i, line in enumerate(lines):
        if re.match(r"^\s+1\.\s+\S", line):
            first_item_idx = i
            break
    if first_item_idx is None:
        print("Could not find start of agenda items, skipping.")
        continue

    # Find where agenda items end
    last_item_idx = len(lines)
    for i in range(len(lines)-1, first_item_idx, -1):
        line = lines[i].strip().lower()
        if ('next' in line) and ('meeting' in line) and ('city' in line) and ('planning' in line) and ('commission' in line):
            last_item_idx = i
            break

    # Keep only agenda items portion
    lines = lines[first_item_idx:last_item_idx]

    # Remove page footers, PAGE BREAK tags, and blankish lines around them
    cleaned = []
    for line in lines:
        stripped = line.strip()

        # skip <PAGE BREAK> tags
        if stripped.lower() == "<page break>":
            continue

        # Skip footer lines
        if re.match(
            r"city planning commission\s+\d+\s+[a-zA-Z]+\s+\d{1,2},\s+\d{4}",
            stripped.lower()
        ):
            continue

        cleaned.append(line)

    # Collapse runs of blank lines to at most 1
    collapsed = []
    prev_blank = False
    for line in cleaned:
        is_blank = (line.strip()=="")
        if is_blank:
            if not prev_blank:
                collapsed.append(line)
            prev_blank = True
        else:
            collapsed.append(line)
            prev_blank = False
    
    # identify lines with top level items
    top_level_indices = [0]
    top_level_indent = collapsed[0].index('1')
    item = 2
    for i, line in enumerate(collapsed):
        # Warn about asterisks near item numbers
        if re.match(r"^\s+\*?" + str(item) + r"\*?\.\s+\S", line):
            if '*' in line.split('.')[0]:
                print(f"WARNING: Asterisk found near item number on line {i}: {line.strip()}")

        if re.match(r"^\s+" + str(item) + r"\.\s+\S", line):
            indent = line.index(str(item))
            if np.abs(indent - top_level_indent) <= 2:
                top_level_indices.append(i)
                item += 1
    
    # Put top level items text into a dict
    out_dict = []
    for i, idx in enumerate(top_level_indices):
        if i < len(top_level_indices)-1:
            item_lines = collapsed[idx:top_level_indices[i+1]]
        else:
            item_lines = collapsed[idx:]
        title_line = item_lines[0]
        item_no = title_line.strip().split('.')[0].strip()

        out_dict.append({
            "item_no": item_no,
            "sub_item_no": "",
            "text": "\n".join(item_lines),
            "is_consent_calendar": bool(re.search(r'\bconsent calendar\b', title_line.lower())),
        })

    # Special handling of consent calendar
    temp_dict = out_dict.copy()
    for k, v in enumerate(temp_dict):
        if v['is_consent_calendar']:
            item_no = v["item_no"]
            text = v["text"]
            lines = text.replace("\r\n", "\n").split("\n")

            # Find where first subitem starts
            first_item_idx = None
            for i, line in enumerate(lines):
                match_by_num_a = re.match(r"^\s+" + str(item_no) + r"a\.\s+\S", line.lower())
                match_by_a = re.match(r"^\s+a\.\s+\S", line.lower())
                if match_by_num_a or match_by_a:
                    first_item_idx = i
                    break
            if first_item_idx is None:
                continue
            top_indent = lines[first_item_idx].index('a')
            
            # Find indices of the subitems
            sub_item_indices = [first_item_idx]
            item = 2
            for i, line in enumerate(lines):
                match_by_num_a = re.match(r"^\s+" + str(item_no) + str(chr(96+item)) + r"\.\s+\S", line.lower())
                match_by_a = re.match(r"^\s+" + str(chr(96+item)) + r"\.\s+\S", line.lower())
                if match_by_num_a or match_by_a:
                    indent = line.index(str(chr(96+item)))
                    if indent == top_indent:
                        sub_item_indices.append(i)
                        item += 1

            out_dict[k]["text"] = "\n".join(lines[0:first_item_idx])
            for i, idx in enumerate(sub_item_indices):
                if i < len(sub_item_indices)-1:
                    item_lines = lines[idx:sub_item_indices[i+1]]
                else:
                    item_lines = lines[idx:]
                title_line = item_lines[0]
                sub_item_no = title_line.strip().split('.')[0].strip()
                
                out_dict.append({
                    "item_no": sub_item_no,
                    "sub_item_no": sub_item_no,
                    "text": "\n".join(item_lines),
                    "is_consent_calendar": True,
                })

    # Save cleaned agenda to json
    with open(output_filepath, 'w', encoding='utf-8') as f:
        sorted_dict = sorted(out_dict, key=sort_key)
        json.dump(sorted_dict, f, indent=2)



0 2018-05-10 ...
1 2018-05-23 ...
2 2018-06-14 ...
3 2018-07-12 ...
4 2018-07-26 ...
5 2018-08-09 ...
6 2018-08-23 ...
7 2018-09-13 ...
8 2018-09-27 ...
9 2018-10-11 ...
10 2018-10-25 ...
11 2018-11-08 ...
12 2018-11-29 ...
13 2018-12-13 ...
14 2018-12-20 ...
15 2019-01-10 ...
16 2019-01-24 ...
17 2019-02-14 ...
18 2019-02-28 ...
19 2019-03-14 ...
20 2019-03-28 ...
21 2019-04-11 ...
22 2019-05-09 ...
23 2019-05-23 ...
24 2019-06-13 ...
25 2019-06-27 ...
26 2019-07-11 ...
27 2019-07-25 ...
28 2019-08-08 ...
29 2019-08-22 ...
30 2019-09-12 ...
31 2019-09-26 ...
32 2019-10-10 ...
33 2019-10-24 ...
34 2019-11-14 ...
35 2019-11-21 ...
36 2019-12-12 ...
37 2019-12-19 ...
38 2020-01-09 ...
39 2020-01-23 ...
40 2020-02-13 ...
41 2020-03-12 ...
42 2020-04-23 ...
43 2020-05-14 ...
44 2020-05-28 ...
45 2020-06-04 ...
46 2020-06-11 ...
47 2020-06-25 ...
48 2020-07-09 ...
49 2020-07-23 ...
50 2020-08-13 ...
51 2020-08-27 ...
52 2020-09-10 ...
53 2020-09-17 ...
54 2020-09-24 ...
55 2020-10-08 ...
56